In [23]:
# Import necessary libraries
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import torch

# Set device to GPU if available, otherwise use CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU available: ", torch.cuda.is_available())

# Add parent directory to path to import functions
sys.path.append(os.path.abspath('..'))
from python.functions import *

GPU available:  True


In [24]:
# Download a test image
from PIL import Image

# Path to the image (to be replaced with actual path if needed)
image_path = '../Data/Balzac.jpg'

# Load the JPG image
img_pil = Image.open(image_path)

# Convert to grayscale and numpy array (0-255)
img_gray = img_pil.convert('L')  # 'L' mode = grayscale
img = np.array(img_gray, dtype=np.uint8)

print(f"Image shape: {img.shape}")
print(f"Value range: {img.min()} to {img.max()}")

Image shape: (593, 528)
Value range: 0 to 255


In [25]:
%matplotlib qt

# Load the saved model
cnn = torch.load('../CNN/dncnn_model.pth')

# Denoise the image using the trained model
cnn.eval()  # Set model to evaluation mode
with torch.no_grad():  # Disable gradient computation for inference
    # Prepare noisy image as model input
    blurred = add_channel(img).unsqueeze(0).to(device)
    
    # Predict noise using DnCNN
    noise_pred = cnn(blurred)
    
    # Reconstruct clean image: noisy - predicted_noise
    denoised = torch.clamp(blurred - noise_pred, 0, 1).cpu().squeeze()

# Visualize: Noisy → Denoised → Original
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.imshow(img, cmap='gray')
plt.title("Noisy Image (Input)")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(denoised, cmap='gray')
plt.title("Denoised Image (Output)")
plt.axis('off')

plt.tight_layout()
plt.show()

C:\Users\adrie\AppData\Local\Temp\ipykernel_12460\531243531.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn = torch.load('../CNN/dncnn_model.pth')
